<a href="https://colab.research.google.com/github/sr606/LLM/blob/main/mermaid_v8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

 ---AFTER normalize----
{
  "stage_id": "ORA_Ext_Vehicle_Off_Road_Data",
  "primary_tables": [],
  "joins": [],
  "filters": [],
  "transformations": []
}

from graphviz import Digraph

def render_drilldown(stages, output_path):

    dot = Digraph("Drilldown", format="pdf")
    dot.attr(rankdir="TB")

    for stage in stages:

        dataset_name = stage["outputs"][0] if stage["outputs"] else "NO_OUTPUT"

        header_id = f"HEADER_{dataset_name}"

        dot.node(header_id,
                 f"DATASET: {dataset_name}",
                 shape="box",
                 style="filled",
                 fillcolor="#90CAF9")

        stage_label = stage["stage_id"] + "\n"

        for join in stage["joins"]:
            stage_label += f"{join['join_type']} {join['table']}\n"
            stage_label += f"ON {join['condition']}\n"

        for t in stage["transformations"]:
            stage_label += f"{t['target']} ← {t['logic']}\n"

        dot.node(stage["stage_id"],
                 stage_label,
                 shape="box",
                 style="rounded,filled",
                 fillcolor="#FFE0B2")

        dot.edge(header_id, stage["stage_id"])

    dot.render(output_path, cleanup=True)

In [ ]:
📁 FINAL CLEAN STABLE STRUCTURE
project/
│
├── main.py
├── agent.py
│
├── parser.py
├── extractor.py
├── llm_normalizer.py
├── validator.py
│
├── renderer_unified.py
├── renderer_drilldown.py
│
├── requirements.txt
│
├── data/input/
├── data/output/
│
└── .env
1️⃣ main.py
from agent import run_agent

if __name__ == "__main__":
    run_agent()
2️⃣ agent.py
import os
from parser import split_into_stages
from extractor import extract_raw_blocks
from llm_normalizer import normalize_stage
from validator import validate_stage
from renderer_unified import render_unified
from renderer_drilldown import render_drilldown

INPUT_FOLDER = "data/input"
OUTPUT_FOLDER = "data/output"


def run_agent():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    for file in os.listdir(INPUT_FOLDER):
        if not file.endswith(".txt"):
            continue

        with open(os.path.join(INPUT_FOLDER, file), "r", encoding="utf-8") as f:
            pseudo = f.read()

        print(f"Processing {file}")

        stages = split_into_stages(pseudo)
        final_stages = []

        for stage in stages:
            raw = extract_raw_blocks(stage["block"])
            normalized = normalize_stage(stage["stage_id"], raw)

            # 🔥 Merge structural info back
            normalized["inputs"] = raw["inputs"]
            normalized["outputs"] = raw["outputs"]

            validated = validate_stage(normalized, raw)
            final_stages.append(validated)

        base = file.replace(".txt", "")

        render_unified(
            final_stages,
            os.path.join(OUTPUT_FOLDER, f"{base}_unified")
        )

        render_drilldown(
            final_stages,
            os.path.join(OUTPUT_FOLDER, f"{base}_dataset_drilldown")
        )

        print("Done.")
3️⃣ parser.py
import re

def split_into_stages(text):

    pattern = r"// --- \[(.*?)\] \[Lines.*?\] ---([\s\S]*?)(?=// --- \[|$)"
    matches = re.findall(pattern, text)

    stages = []

    for header, body in matches:
        name_match = re.search(r":\s*(.*)", header)
        if not name_match:
            continue

        stage_id = name_match.group(1).strip()

        stages.append({
            "stage_id": stage_id,
            "block": body.strip()
        })

    return stages
4️⃣ extractor.py (FIXED)
import re

def extract_raw_blocks(block):

    # SQL extraction
    sql_match = re.search(r"SQL:(.*?)(StageType:|Output:|$)", block, re.DOTALL)
    sql_block = sql_match.group(1) if sql_match else ""

    # 🔥 FIXED transformation extraction (capture full block)
    transform_match = re.search(
        r"Transformations:(.*)",
        block,
        re.DOTALL
    )
    transform_block = transform_match.group(1) if transform_match else ""

    input_datasets = re.findall(
        r"Input:\s*←\s*dataset_\d+\s*\(([^)]+)\)", block
    )

    output_datasets = re.findall(
        r"Output:\s*→\s*dataset_\d+\s*\(([^)]+)\)", block
    )

    return {
        "sql": sql_block.strip(),
        "transform": transform_block.strip(),
        "inputs": input_datasets,
        "outputs": output_datasets
    }
5️⃣ llm_normalizer.py (FIXED PRIMARY TABLE + LIMITS)
import os
import json
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")


def normalize_stage(stage_id, raw):

    prompt = f"""
Extract structured lineage.

Return STRICT JSON only:

{{
  "stage_id": "{stage_id}",
  "primary_tables": [],
  "joins": [
    {{
      "join_type": "",
      "table": "",
      "condition": ""
    }}
  ],
  "filters": [],
  "transformations": [
    {{
      "target": "",
      "logic": ""
    }}
  ]
}}

Rules:
- Extract top-level primary table from first FROM
- Extract max 3 joins
- Extract max 6 transformations
- Ignore nested subqueries
- Copy text exactly from input
"""

    response = client.chat.completions.create(
        model=DEPLOYMENT,
        temperature=0.0,
        messages=[
            {"role": "system", "content": "You extract SQL lineage strictly."},
            {"role": "user", "content": prompt + "\nSQL:\n" + raw["sql"] + "\nTRANSFORM:\n" + raw["transform"]}
        ]
    )

    content = response.choices[0].message.content.strip()

    print("\nLLM RAW RESPONSE:\n", content)

    try:
        return json.loads(content)
    except:
        return {
            "stage_id": stage_id,
            "primary_tables": [],
            "joins": [],
            "filters": [],
            "transformations": []
        }
6️⃣ validator.py (SAFE LIMITING)
def validate_stage(normalized, raw):

    sql_text = raw["sql"]
    transform_text = raw["transform"]

    # Validate joins
    valid_joins = []
    for j in normalized.get("joins", []):
        if j["condition"] in sql_text:
            valid_joins.append(j)

    normalized["joins"] = valid_joins[:3]

    # Validate transformations
    valid_transforms = []
    for t in normalized.get("transformations", []):
        if t["logic"] in transform_text:
            valid_transforms.append(t)

    normalized["transformations"] = valid_transforms[:6]

    return normalized
7️⃣ renderer_unified.py
from graphviz import Digraph

def render_unified(stages, output_path):

    dot = Digraph("Unified", format="pdf")
    dot.attr(rankdir="LR")

    previous_dataset = None

    for stage in stages:

        label = stage["stage_id"] + "\n"

        # Primary table
        if stage["primary_tables"]:
            src_label = "\n".join(stage["primary_tables"])
            dot.node(stage["stage_id"]+"_src",
                     src_label,
                     shape="box",
                     style="filled",
                     fillcolor="#BBDEFB")
            dot.edge(stage["stage_id"]+"_src", stage["stage_id"])

        # Joins
        for join in stage["joins"]:
            label += f"{join['join_type']} {join['table']}\n"
            label += f"ON {join['condition']}\n"

        dot.node(stage["stage_id"],
                 label,
                 shape="box",
                 style="rounded,filled",
                 fillcolor="#FFE0B2")

        if previous_dataset:
            dot.edge(previous_dataset, stage["stage_id"])

        if stage["outputs"]:
            ds = stage["outputs"][0]
            dot.node(ds, ds, shape="cylinder")
            dot.edge(stage["stage_id"], ds)
            previous_dataset = ds

    dot.render(output_path, cleanup=True)
8️⃣ renderer_drilldown.py (HEADER BOX VERSION)
from graphviz import Digraph

def render_drilldown(stages, output_path):

    dot = Digraph("Drilldown", format="pdf")
    dot.attr(rankdir="TB")

    for stage in stages:

        dataset_name = stage["outputs"][0] if stage["outputs"] else "NO_OUTPUT"

        header_id = f"HEADER_{dataset_name}"

        dot.node(header_id,
                 f"DATASET: {dataset_name}",
                 shape="box",
                 style="filled",
                 fillcolor="#90CAF9")

        stage_label = stage["stage_id"] + "\n"

        for join in stage["joins"]:
            stage_label += f"{join['join_type']} {join['table']}\n"
            stage_label += f"ON {join['condition']}\n"

        for t in stage["transformations"]:
            stage_label += f"{t['target']} ← {t['logic']}\n"

        dot.node(stage["stage_id"],
                 stage_label,
                 shape="box",
                 style="rounded,filled",
                 fillcolor="#FFE0B2")

        dot.edge(header_id, stage["stage_id"])

    dot.render(output_path, cleanup=True)


In [ ]:
Processing Sample_Job1 1 2_detailed_pseudocode.txt

LLM RAW RESPONSE:
 ```json
{
  "stage_id": "ORA_Ext_Vehicle_Off_Road_Data",
  "primary_tables": [
    "supplierdata_ext.vor_vehicle_off_road"
  ],
  "joins": [
    {
      "join_type": "LEFT OUTER JOIN",
      "table": "smr_owner.d_vor_progress",
      "condition": "vor.original_worksheet_number = vnrr.original_worksheet_number and VNRR.LATEST_FLAG='Y'"
    },
    {
      "join_type": "LEFT OUTER JOIN",
      "table": "cdm_owner.vehicle_dim",
      "condition": "vor.registration_number = v.registration_no AND VOR.NOLS_CUSTOMER_NUMBER=V.CUST_ID and TO_DATE (VOR.INCIDENT_CREATION_DATE, 'DD/MM/YYYY') between v.EFFECTIVE_START_DATE and V.EFFECTIVE_END_DATE"
    },
    {
      "join_type": "LEFT OUTER JOIN",
      "table": "cdm_owner.customer_dim",
      "condition": "cust.cust_id = vor.nols_customer_number AND cust.latest_flag = 'Y'"
    }
  ],
  "filters": [
    "VOR.LAST_UPDATED_ON>=TO_DATE('#PREV_BATCH_RUN_DATE_VOR#','YYYY-MM-DD')-#DELTA_DAYS_PARAM#"
  ],
  "transformations": [
    {
      "target": "REGISTERED_CUST_DRIVER_SK",
      "logic": "NVL (cd.cust_driver_sk, -99)"
    },
    {
      "target": "SUPP_SK",
      "logic": "(Case WHEN CAPTURED_garage_name is null then -98 when supp_smr.supp_sk is not null and supp_smr.lkp_key='A' then supp_smr.supp_sk when supp_nols.supp_sk is not null AND supp_nols.lkp_key='A' then supp_nols.supp_sk when supp_smr.supp_sk is not null and supp_smr.lkp_key='B' then -97 when supp_nols.supp_sk is not null and supp_nols.lkp_key='B' then -97 Else -99 end)"
    },
    {
      "target": "ORIGINAL_WORKSHEET_NUMBER",
      "logic": "NVL (vor.original_worksheet_number,'N/A')"
    },
    {
      "target": "vehicle_sk",
      "logic": "(CASE WHEN v.vehicle_sk IS NOT NULL THEN v.vehicle_sk WHEN VN.VEHICLE_SK IS NOT NULL THEN VN.VEHICLE_SK ELSE -99 END)"
    },
    {
      "target": "variant_sk",
      "logic": "(CASE WHEN v.variant_sk IS NOT NULL THEN v.variant_sk WHEN VN.variant_sk IS NOT NULL THEN VN.variant_sk ELSE -99 END)"
    },
    {
      "target": "JOB_CLOSED_DATE_SK",
      "logic": "CASE WHEN vor.work_finish_date IS NULL THEN '99991231' WHEN vor.work_finish_date='In Progress' THEN '99991231' ELSE TO_CHAR (TO_DATE (vor.work_finish_date, 'DD/MM/YYYY'), 'YYYYMMDD') END"
    }
  ]
}
```

LLM RAW RESPONSE:
 ```json
{
  "stage_id": "Ora_StgVehicleOffRoad",
  "primary_tables": [],
  "joins": [
    {
      "join_type": "",
      "table": "",
      "condition": ""
    }
  ],
  "filters": [],
  "transformations": [
    {
      "target": "",
      "logic": ""
    }
  ]
}
```

LLM RAW RESPONSE:
 ```json
{
  "stage_id": "HF_FACT_VOR_DATA",
  "primary_tables": [],
  "joins": [
    {
      "join_type": "",
      "table": "",
      "condition": ""
    }
  ],
  "filters": [],
  "transformations": [
    {
      "target": "",
      "logic": ""
    }
  ]
}
```

LLM RAW RESPONSE:
 ```json
{
  "stage_id": "Tfm_LoadRecords",
  "primary_tables": [
    "Lnk_Vehicle_off_Road_Data"
  ],
  "joins": [
    {
      "join_type": "",
      "table": "lkVehicleDim_REg",
      "condition": "Lnk_Vehicle_off_Road_Data.VEHICLE_SK=-99"
    }
  ],
  "filters": [],
  "transformations": [
    {
      "target": "VEHICLE_SK",
      "logic": "If Lnk_Vehicle_off_Road_Data.VEHICLE_SK=-99 then lkVehicleDim_REg.VEHICLE_SK else Lnk_Vehicle_off_Road_Data.VEHICLE_SK"
    },
    {
      "target": "VARIANT_SK",
      "logic": "If Lnk_Vehicle_off_Road_Data.VEHICLE_SK=-99 then lkVehicleDim_REg.VARIANT_SK else Lnk_Vehicle_off_Road_Data.VARIANT_SK"
    },
    {
      "target": "SUPP_SK",
      "logic": "\\(20)Lnk_Vehicle_off_Road_Data.SUPP_SK"
    },
    {
      "target": "DESCRIPTION",
      "logic": "If Lnk_Vehicle_off_Road_Data.SUPP_SK=-99 then 'Unknown' Else 'Duplicates'"
    }
  ]
}
```

LLM RAW RESPONSE:
 ```json
{
  "stage_id": "HF_SMR_VEHICLE_DIM",
  "primary_tables": [],
  "joins": [],
  "filters": [],
  "transformations": [
    {
      "target": "REGISTRATION_NO",
      "logic": "Lnk_Vehicle_off_Road_Data.VOR_REGISTRATION_NUMBER"
    }
  ]
}
```

LLM RAW RESPONSE:
 ```json
{
  "stage_id": "Supp_VOR_Exception",
  "primary_tables": [],
  "joins": [
    {
      "join_type": "",
      "table": "",
      "condition": ""
    }
  ],
  "filters": [],
  "transformations": [
    {
      "target": "",
      "logic": ""
    }
  ]
}
```
Done.

In [ ]:
✅ FINAL renderer_unified.py (HTML VERSION)

Replace entire file with this:

from graphviz import Digraph
import textwrap


def wrap_text(text, width=80):
    lines = textwrap.wrap(text, width=width, break_long_words=False)
    return "<BR/>".join(lines)


def render_unified(stages, output_path):

    dot = Digraph("Unified", format="pdf")
    dot.attr(rankdir="LR", splines="spline")

    previous_dataset = None

    for stage in stages:

        stage_id = stage["stage_id"]

        # ----------------------------
        # Build HTML Label
        # ----------------------------
        rows = []

        # Stage title
        rows.append(
            f'<TR><TD BGCOLOR="#FFE0B2"><B>{stage_id}</B></TD></TR>'
        )

        # Primary tables
        if stage.get("primary_tables"):
            rows.append('<TR><TD ALIGN="LEFT"><B>PRIMARY TABLE:</B></TD></TR>')
            for table in stage["primary_tables"]:
                rows.append(
                    f'<TR><TD ALIGN="LEFT">{wrap_text(table)}</TD></TR>'
                )

        # Joins
        if stage.get("joins"):
            rows.append('<TR><TD ALIGN="LEFT"><B>JOINS:</B></TD></TR>')
            for join in stage["joins"]:
                join_text = (
                    f"{join['join_type']} {join['table']}<BR/>"
                    f"ON {wrap_text(join['condition'])}"
                )
                rows.append(
                    f'<TR><TD ALIGN="LEFT">{join_text}</TD></TR>'
                )

        # Filters
        if stage.get("filters"):
            rows.append('<TR><TD ALIGN="LEFT"><B>FILTERS:</B></TD></TR>')
            for f in stage["filters"]:
                rows.append(
                    f'<TR><TD ALIGN="LEFT">{wrap_text(f)}</TD></TR>'
                )

        # Transformations
        if stage.get("transformations"):
            rows.append('<TR><TD ALIGN="LEFT"><B>TRANSFORMATIONS:</B></TD></TR>')
            for t in stage["transformations"]:
                transform_text = (
                    f"{t['target']} ← {wrap_text(t['logic'])}"
                )
                rows.append(
                    f'<TR><TD ALIGN="LEFT">{transform_text}</TD></TR>'
                )

        html_label = f"""<
        <TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" WIDTH="600">
            {''.join(rows)}
        </TABLE>
        >"""

        dot.node(stage_id,
                 label=html_label,
                 shape="plaintext")

        # Connect datasets
        if previous_dataset:
            dot.edge(previous_dataset, stage_id)

        if stage.get("outputs"):
            ds = stage["outputs"][0]
            dot.node(ds,
                     f"""<
                     <TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0">
                     <TR><TD BGCOLOR="#BBDEFB"><B>{ds}</B></TD></TR>
                     </TABLE>
                     >""",
                     shape="plaintext")

            dot.edge(stage_id, ds)
            previous_dataset = ds

    dot.render(output_path, cleanup=True)
✅ FINAL renderer_drilldown.py (HEADER BOX VERSION)

Replace entire file with:

from graphviz import Digraph
import textwrap


def wrap_text(text, width=80):
    lines = textwrap.wrap(text, width=width, break_long_words=False)
    return "<BR/>".join(lines)


def render_drilldown(stages, output_path):

    dot = Digraph("Drilldown", format="pdf")
    dot.attr(rankdir="TB", splines="spline")

    for stage in stages:

        dataset_name = stage["outputs"][0] if stage["outputs"] else "NO_OUTPUT"

        # ----------------------------
        # Dataset Header Box
        # ----------------------------
        header_id = f"HEADER_{dataset_name}"

        header_label = f"""<
        <TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" WIDTH="700">
            <TR>
                <TD BGCOLOR="#90CAF9">
                    <B>DATASET: {dataset_name}</B>
                </TD>
            </TR>
        </TABLE>
        >"""

        dot.node(header_id,
                 label=header_label,
                 shape="plaintext")

        # ----------------------------
        # Stage Detail Box
        # ----------------------------
        rows = []

        rows.append(
            f'<TR><TD BGCOLOR="#FFE0B2"><B>{stage["stage_id"]}</B></TD></TR>'
        )

        if stage.get("primary_tables"):
            rows.append('<TR><TD ALIGN="LEFT"><B>PRIMARY TABLE:</B></TD></TR>')
            for table in stage["primary_tables"]:
                rows.append(
                    f'<TR><TD ALIGN="LEFT">{wrap_text(table)}</TD></TR>'
                )

        if stage.get("joins"):
            rows.append('<TR><TD ALIGN="LEFT"><B>JOINS:</B></TD></TR>')
            for join in stage["joins"]:
                join_text = (
                    f"{join['join_type']} {join['table']}<BR/>"
                    f"ON {wrap_text(join['condition'])}"
                )
                rows.append(
                    f'<TR><TD ALIGN="LEFT">{join_text}</TD></TR>'
                )

        if stage.get("filters"):
            rows.append('<TR><TD ALIGN="LEFT"><B>FILTERS:</B></TD></TR>')
            for f in stage["filters"]:
                rows.append(
                    f'<TR><TD ALIGN="LEFT">{wrap_text(f)}</TD></TR>'
                )

        if stage.get("transformations"):
            rows.append('<TR><TD ALIGN="LEFT"><B>TRANSFORMATIONS:</B></TD></TR>')
            for t in stage["transformations"]:
                transform_text = (
                    f"{t['target']} ← {wrap_text(t['logic'])}"
                )
                rows.append(
                    f'<TR><TD ALIGN="LEFT">{transform_text}</TD></TR>'
                )

        stage_label = f"""<
        <TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" WIDTH="700">
            {''.join(rows)}
        </TABLE>
        >"""

        dot.node(stage["stage_id"],
                 label=stage_label,
                 shape="plaintext")

        dot.edge(header_id, stage["stage_id"])

    dot.render(output_path, cleanup=True)

In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")


def generate_dot(workflow_name, stages):

    prompt = f"""
You are generating Graphviz DOT code.

Rules:
- Output ONLY valid DOT code.
- No markdown.
- No explanations.
- Use digraph.
- rankdir=LR.
- Use subgraph cluster_workflow.
- Use HTML table nodes.
- Preserve ALL joins and transformations.
- Do NOT invent new stages.
- Use exact stage_id names.
- Connect stages using outputs to inputs.
- Style similar to Alteryx workflow.
"""

    response = client.chat.completions.create(
        model=DEPLOYMENT,
        temperature=0.0,
        messages=[
            {"role": "system", "content": "You generate Graphviz DOT workflows."},
            {
                "role": "user",
                "content": prompt + "\n\nWorkflow JSON:\n" + str({
                    "workflow_name": workflow_name,
                    "stages": stages
                })
            }
        ]
    )

    return response.choices[0].message.content.strip()




from dot_generator import generate_dot
import graphviz

dot_code = generate_dot(base, final_stages)

print("\nGENERATED DOT:\n")
print(dot_code)

src = graphviz.Source(dot_code)
src.render(f"data/output/{base}_unified", format="pdf", cleanup=True)


# import os
# from parser import split_into_stages
# from extractor import extract_raw_blocks
# from llm_normalizer import normalize_stage
# from validator import validate_stage
# # from renderer_unified import render_unified
# # from renderer_drilldown import render_drilldown

# INPUT_FOLDER = "data/input"
# OUTPUT_FOLDER = "data/output"


# def run_agent():
#     os.makedirs(OUTPUT_FOLDER, exist_ok=True)

#     for file in os.listdir(INPUT_FOLDER):
#         if not file.endswith(".txt"):
#             continue

#         with open(os.path.join(INPUT_FOLDER, file), "r", encoding="utf-8") as f:
#             pseudo = f.read()

#         print(f"Processing {file}")

#         stages = split_into_stages(pseudo)
#         final_stages = []

#         for stage in stages:
#             raw = extract_raw_blocks(stage["block"])
#             normalized = normalize_stage(stage["stage_id"], raw)

#             # 🔥 Merge structural info back
#             normalized["inputs"] = raw["inputs"]
#             normalized["outputs"] = raw["outputs"]

#             validated = validate_stage(normalized, raw)
#             final_stages.append(validated)

#         base = file.replace(".txt", "")

#         # render_unified(
#         #     final_stages,
#         #     os.path.join(OUTPUT_FOLDER, f"{base}_unified")
#         # )

#         # render_drilldown(
#         #     final_stages,
#         #     os.path.join(OUTPUT_FOLDER, f"{base}_dataset_drilldown")
#         # )

#         print("Done.")

# from dot_generator import generate_dot
# import graphviz

# dot_code = generate_dot(base, final_stages)

# print("\nGENERATED DOT:\n")
# print(dot_code)

# src = graphviz.Source(dot_code)
# src.render(f"data/output/{base}_unified", format="pdf", cleanup=True)


import os
from parser import split_into_stages
from extractor import extract_raw_blocks
from llm_normalizer import normalize_stage
from validator import validate_stage
from dot_generator import generate_dot

import graphviz  # requires Graphviz system install (dot.exe) on PATH

INPUT_FOLDER = "data/input"
OUTPUT_FOLDER = "data/output"


def process_file(file_path: str, file_name: str):
    """
    Processes a single pseudocode .txt file:
      - splits into stages
      - extracts raw blocks
      - normalizes/validates
      - generates DOT
      - renders PDF diagram
    """
    with open(file_path, "r", encoding="utf-8") as f:
        pseudo = f.read()

    print(f"Processing: {file_name}")

    stages = split_into_stages(pseudo)
    final_stages = []

    for stage in stages:
        # 1) Extract raw blocks from the stage block text
        raw = extract_raw_blocks(stage["block"])

        # 2) Normalize with the LLM layer
        normalized = normalize_stage(stage["stage_id"], raw)

        # 3) Merge structural info (inputs/outputs) back to normalized
        normalized["inputs"] = raw.get("inputs", [])
        normalized["outputs"] = raw.get("outputs", [])

        # 4) Validate the stage vs raw
        validated = validate_stage(normalized, raw)
        final_stages.append(validated)

    # Use base name to name outputs
    base = file_name.replace(".txt", "")

    # 5) Generate Graphviz DOT code for the final stages
    dot_code = generate_dot(base, final_stages)

    print("\nGENERATED DOT:\n")
    print(dot_code)

    # 6) Render the DOT to PDF
    output_base_path = os.path.join(OUTPUT_FOLDER, f"{base}_unified")
    src = graphviz.Source(dot_code)
    # format can be "pdf", "png", "svg"
    src.render(output_base_path, format="pdf", cleanup=True)

    print(f"Rendered: {output_base_path}.pdf\n")


def run_agent():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    if not os.path.isdir(INPUT_FOLDER):
        raise FileNotFoundError(f"Input folder not found: {INPUT_FOLDER}")

    files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".txt")]
    if not files:
        print(f"No .txt files found in {INPUT_FOLDER}")
        return

    for file in files:
        file_path = os.path.join(INPUT_FOLDER, file)
        try:
            process_file(file_path, file)
        except Exception as e:
            # Don’t fail the whole run for one file; log and continue
            print(f"[ERROR] Failed to process {file}: {e}")

    print("Done.")


if __name__ == "__main__":
    # Optional inline workaround if Graphviz isn’t on PATH globally:
    # import os
    # os.environ["GRAPHVIZ_DOT"] = r"C:\Program Files\Graphviz\bin\dot.exe"
    # or prepend PATH:
    # gv_bin = r"C:\Program Files\Graphviz\bin"
    # os.environ["PATH"] = gv_bin + os.pathsep + os.environ.get("PATH", "")

    run_agent()



    # import os
# from dotenv import load_dotenv
# from openai import AzureOpenAI

# load_dotenv()

# client = AzureOpenAI(
#     api_key=os.getenv("AZURE_OPENAI_API_KEY"),
#     api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
#     azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
# )

# DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")


# def generate_dot(workflow_name, stages):

#     prompt = f"""
# You are generating Graphviz DOT code.

# Rules:
# - Output ONLY valid DOT code.
# - No markdown.
# - No explanations.
# - Use digraph.
# - rankdir=LR.
# - Use subgraph cluster_workflow.
# - Use HTML table nodes.
# - Preserve ALL joins and transformations.
# - Do NOT invent new stages.
# - Use exact stage_id names.
# - Connect stages using outputs to inputs.
# - Style similar to Alteryx workflow.
# """

#     response = client.chat.completions.create(
#         model=DEPLOYMENT,
#         temperature=0.0,
#         messages=[
#             {"role": "system", "content": "You generate Graphviz DOT workflows."},
#             {
#                 "role": "user",
#                 "content": prompt + "\n\nWorkflow JSON:\n" + str({
#                     "workflow_name": workflow_name,
#                     "stages": stages
#                 })
#             }
#         ]
#     )

#     return response.choices[0].message.content.strip()


import os
import re
import json
import html
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")


_PROMPT_RULES = """You are generating Graphviz DOT code.

Strict rules (must follow exactly):
- Output ONLY raw DOT code. No markdown, no triple backticks, no surrounding prose.
- The first non-whitespace token must be: digraph
- Use: rankdir=LR
- Wrap all content in: subgraph cluster_workflow { ... } with a label equal to the workflow name.
- Use HTML-like labels with: label=< ... > and node [shape=plaintext].
- IMPORTANT: Use literal '<' and '>' inside label=<...>. DO NOT HTML-escape (&lt; &gt; &amp;).
- Preserve ALL stage_ids exactly as provided (no renaming).
- Do NOT invent or omit stages.
- Connect stages using known outputs->inputs from the provided stages.
- Sanitize identifiers (graph id, node ids): only letters/digits/underscore. Replace spaces and other symbols with '_'.
- No comments unless they are valid DOT comments (// or /* */), but prefer none.
"""


def _strip_code_fences(s: str) -> str:
    """Remove any Markdown code fences if present."""
    # Remove any leading/trailing ```... fences (language optional)
    # Use a conservative approach across the whole text.
    fence_pattern = re.compile(r"^\s*```[a-zA-Z0-9]*\s*|\s*```\s*$", re.MULTILINE)
    s = fence_pattern.sub("", s)
    return s.strip()


def _extract_dot_block(s: str) -> str:
    """
    From a possibly noisy response, extract the DOT block that starts with
    'digraph' or 'graph' and ends at the last closing brace.
    """
    s = s.strip()
    # Find the start of the graph
    m = re.search(r"(digraph|graph)\s+[^\{]*\{", s, flags=re.DOTALL)
    if not m:
        return s  # best effort; sanity check will catch this later
    start = m.start()

    # Heuristic: take everything up to the last '}' in the text
    end = s.rfind("}")
    if end == -1:
        return s[start:].strip()
    return s[start:end + 1].strip()


def _clean_and_validate(dot_text: str) -> str:
    """
    - Strip code fences
    - Unescape HTML entities to literal characters
    - Extract only the DOT block
    - Validate it begins with digraph/graph
    """
    # 1) Strip code fences
    dot_text = _strip_code_fences(dot_text)

    # 2) Unescape &lt; &gt; &amp; etc.
    dot_text = html.unescape(dot_text)

    # 3) Extract only the DOT block
    dot_text = _extract_dot_block(dot_text)

    # 4) Final sanity checks
    if not dot_text.lstrip().startswith(("digraph", "graph")):
        raise ValueError("DOT must start with 'digraph' or 'graph'. Got:\n" + dot_text[:120])

    # Quick guard against accidental markdown remnants
    if "```" in dot_text or "LLM RAW RESPONSE" in dot_text:
        raise ValueError("DOT contains unexpected markdown/log text. Clean the generator output.")

    return dot_text


def generate_dot(workflow_name, stages):
    """
    Returns clean DOT text, safe to feed to graphviz.Source(...).
    """
    # Build a clean JSON payload for the model (avoid Python dict repr)
    payload = json.dumps({
        "workflow_name": workflow_name,
        "stages": stages
    }, ensure_ascii=False)

    messages = [
        {"role": "system", "content": "You generate strictly valid Graphviz DOT workflows."},
        {"role": "user", "content": _PROMPT_RULES + "\n\nWorkflow JSON:\n" + payload}
    ]

    response = client.chat.completions.create(
        model=DEPLOYMENT,
        temperature=0.0,
        messages=messages,
        # Optional: discourage markdown fences
        stop=["```"]
    )

    raw = response.choices[0].message.content or ""
    cleaned = _clean_and_validate(raw)

    # Optional: extra assert for HTML-like labels usage
    # Not strictly required, but useful to ensure <table> labels pass through.
    # if "&lt;" in cleaned or "&gt;" in cleaned:
    #     raise ValueError("Found HTML-escaped angle brackets. The DOT must use literal '<' and '>' in label=<...>.")

    return cleaned

In [ ]:
🏗 Step 1 — New LLM Function (Stage-Level Only)

Create new file:

llm_stage_formatter.py
import os
import json
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")


PROMPT = """
You are formatting ONE ETL stage for Graphviz HTML label.

STRICT RULES:
- Return ONLY HTML table rows (no <table> wrapper).
- Do NOT return markdown.
- Do NOT return explanations.
- Do NOT rename stage_id.
- Preserve all joins, filters, transformations.
- Keep full join conditions.
- Format clean and readable using <BR/> where needed.

Sections order:
1) PRIMARY TABLES
2) JOINS
3) FILTERS
4) TRANSFORMATIONS

Return HTML rows only like:
<TR><TD>...</TD></TR>
"""


def format_stage(stage_json):

    payload = json.dumps(stage_json, ensure_ascii=False)

    response = client.chat.completions.create(
        model=DEPLOYMENT,
        temperature=0.0,
        messages=[
            {"role": "system", "content": "You format ETL content cleanly."},
            {"role": "user", "content": PROMPT + "\n\nStage JSON:\n" + payload}
        ],
    )

    return response.choices[0].message.content.strip()
🏗 Step 2 — Deterministic DOT Builder

Create:

dot_builder.py
from graphviz import Digraph


def build_vertical_workflow(stages, output_path):

    dot = Digraph("Workflow", format="pdf")
    dot.attr(rankdir="TB")
    dot.node_attr.update(shape="plaintext")

    previous_node = None

    for stage in stages:

        stage_id = stage["stage_id"]

        html_label = f"""
        <
        <TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" WIDTH="900">
            <TR>
                <TD BGCOLOR="#FFE0B2"><B>{stage_id}</B></TD>
            </TR>
            {stage["html_rows"]}
        </TABLE>
        >
        """

        dot.node(stage_id, label=html_label)

        if previous_node:
            dot.edge(previous_node, stage_id)

        previous_node = stage_id

    dot.render(output_path, cleanup=True)
🏗 Step 3 — Modify agent.py

Replace DOT generation logic with:

from llm_stage_formatter import format_stage
from dot_builder import build_vertical_workflow

# After parsing & validating stages

for stage in final_stages:
    stage["html_rows"] = format_stage(stage)

build_vertical_workflow(
    final_stages,
    f"data/output/{base}_vertical"
)


In [ ]:
===== STAGE JSON SENT TO LLM======
[
  {
    "stage_id": "HF_FACT_VOR_DATA",
    "primary_tables": [],
    "joins": [],
    "filters": [],
    "transformations": [],
    "inputs": [
      "Tfm_LoadRecords"
    ],
    "outputs": [
      "HF_FACT_VOR_DATA"
    ],
    "html_rows": "```html\n<TR><TD>stage_id: HF_FACT_VOR_DATA</TD></TR>\n<TR><TD><B>PRIMARY TABLES</B></TD></TR>\n<TR><TD>Tfm_LoadRecords</TD></TR>\n<TR><TD><B>JOINS</B></TD></TR>\n<TR><TD>No joins specified</TD></TR>\n<TR><TD><B>FILTERS</B></TD></TR>\n<TR><TD>No filters specified</TD></TR>\n<TR><TD><B>TRANSFORMATIONS</B></TD></TR>\n<TR><TD>No transformations specified</TD></TR>\n```"
  },
  {
    "stage_id": "HF_SMR_VEHICLE_DIM",
    "primary_tables": [],
    "joins": [],
    "filters": [],
    "transformations": [],
    "inputs": [],
    "outputs": [
      "HF_SMR_VEHICLE_DIM"
    ],
    "html_rows": "```html\n<TR><TD><B>Stage ID:</B> HF_SMR_VEHICLE_DIM</TD></TR>\n<TR><TD><B>PRIMARY TABLES</B></TD></TR>\n<TR><TD>No primary tables</TD></TR>\n<TR><TD><B>JOINS</B></TD></TR>\n<TR><TD>No joins</TD></TR>\n<TR><TD><B>FILTERS</B></TD></TR>\n<TR><TD>No filters</TD></TR>\n<TR><TD><B>TRANSFORMATIONS</B></TD></TR>\n<TR><TD>No transformations</TD></TR>\n```"
  },
  {
    "stage_id": "ORA_Ext_Vehicle_Off_Road_Data",
    "primary_tables": [],
    "joins": [],
    "filters": [],
    "transformations": [],
    "inputs": [],
    "outputs": [
      "ORA_Ext_Vehicle_Off_Road_Data"
    ],
    "html_rows": "```html\n<TR><TD><B>Stage ID:</B> ORA_Ext_Vehicle_Off_Road_Data</TD></TR>\n<TR><TD><B>PRIMARY TABLES</B></TD></TR>\n<TR><TD>No primary tables</TD></TR>\n<TR><TD><B>JOINS</B></TD></TR>\n<TR><TD>No joins</TD></TR>\n<TR><TD><B>FILTERS</B></TD></TR>\n<TR><TD>No filters</TD></TR>\n<TR><TD><B>TRANSFORMATIONS</B></TD></TR>\n<TR><TD>No transformations</TD></TR>\n```"
  },
  {
    "stage_id": "Ora_StgVehicleOffRoad",
    "primary_tables": [],
    "joins": [],
    "filters": [],
    "transformations": [],
    "inputs": [
      "HF_FACT_VOR_DATA"
    ],
    "outputs": [
      "Ora_StgVehicleOffRoad"
    ],
    "html_rows": "```html\n<TR><TD><B>Stage ID:</B> Ora_StgVehicleOffRoad</TD></TR>\n<TR><TD><B>PRIMARY TABLES</B></TD></TR>\n<TR><TD>HF_FACT_VOR_DATA</TD></TR>\n<TR><TD><B>JOINS</B></TD></TR>\n<TR><TD>No joins specified</TD></TR>\n<TR><TD><B>FILTERS</B></TD></TR>\n<TR><TD>No filters specified</TD></TR>\n<TR><TD><B>TRANSFORMATIONS</B></TD></TR>\n<TR><TD>No transformations specified</TD></TR>\n```"
  },
  {
    "stage_id": "Supp_VOR_Exception",
    "primary_tables": [],
    "joins": [],
    "filters": [],
    "transformations": [],
    "inputs": [
      "Tfm_LoadRecords"
    ],
    "outputs": [
      "Supp_VOR_Exception"
    ],
    "html_rows": "```html\n<TR><TD><B>Stage ID:</B> Supp_VOR_Exception</TD></TR>\n<TR><TD><B>PRIMARY TABLES</B></TD></TR>\n<TR><TD>None</TD></TR>\n<TR><TD><B>JOINS</B></TD></TR>\n<TR><TD>None</TD></TR>\n<TR><TD><B>FILTERS</B></TD></TR>\n<TR><TD>None</TD></TR>\n<TR><TD><B>TRANSFORMATIONS</B></TD></TR>\n<TR><TD>None</TD></TR>\n```"
  },
  {
    "stage_id": "Tfm_LoadRecords",
    "primary_tables": [],
    "joins": [],
    "filters": [],
    "transformations": [],
    "inputs": [
      "ORA_Ext_Vehicle_Off_Road_Data",
      "HF_SMR_VEHICLE_DIM"
    ],
    "outputs": [
      "Tfm_LoadRecords"
    ]
  }
]

==================================
Rendered vertical workflow at base: data/output/Sample_Job1 1 2_detailed_pseudocode_vertical